# Рекуррентные нейронные сети

В предыдущем модуле мы рассмотрели богатые семантические представления текста. Архитектура, которую мы использовали, захватывает агрегированный смысл слов в предложении, но не учитывает **порядок** слов, потому что операция агрегации после эмбеддингов удаляет эту информацию из исходного текста. Так как эти модели не в состоянии представить порядок слов, они не могут решать более сложные или неоднозначные задачи, такие как генерация текста или ответы на вопросы.

Чтобы захватывать смысл последовательности текста, мы будем использовать архитектуру нейронной сети, называемую **рекуррентной нейронной сетью**, или RNN. При использовании RNN мы пропускаем наше предложение через сеть по одному токену за раз, и сеть производит некоторое **состояние**, которое затем передаётся в сеть вместе со следующим токеном.

![Изображение, показывающее пример генерации рекуррентной нейронной сетью.](../../../../../translated_images/ru/rnn.27f5c29c53d727b5.webp)

Имея входную последовательность токенов $X_0,\dots,X_n$, RNN создаёт последовательность блоков нейронной сети и обучает эту последовательность сквозным распространением ошибки. Каждый блок сети принимает пару $(X_i,S_i)$ на вход и выдаёт $S_{i+1}$ в результате. Финальное состояние $S_n$ или выход $Y_n$ поступают в линейный классификатор для получения результата. Все блоки сети используют одинаковые веса и обучаются сквозным проходом.

> На рисунке выше показана рекуррентная нейронная сеть в развернутом виде (слева) и в более компактном рекуррентном представлении (справа). Важно понимать, что все ячейки RNN имеют одинаковые **совместно используемые веса**.

Поскольку в сеть передаются векторы состояний $S_0,\dots,S_n$, RNN способна обучаться последовательным зависимостям между словами. Например, когда в последовательности появляется слово *not* (не), сеть может научиться отрицать определённые элементы в векторе состояния.

Внутри каждая ячейка RNN содержит две матрицы весов: $W_H$ и $W_I$, а также смещение $b$. На каждом шаге RNN, имея на входе $X_i$ и состояние $S_i$, выходное состояние вычисляется как $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, где $f$ — функция активации (часто $\tanh$).

> Для задач типа генерации текста (которые мы рассмотрим в следующем разделе) или машинного перевода мы также хотим получать некоторое выходное значение на каждом шаге RNN. В этом случае добавляется ещё одна матрица $W_O$, и выход вычисляется как $Y_i=f(W_O\times S_i+b_O)$.

Давайте посмотрим, как рекуррентные нейронные сети могут помочь нам классифицировать наш набор новостей.

> Для песочницы нужно запустить следующую ячейку, чтобы убедиться, что необходимая библиотека установлена, а данные предзагружены. Если вы запускаете локально, эту ячейку можно пропустить.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

При обучении больших моделей выделение памяти GPU может стать проблемой. Также нам возможно придется экспериментировать с разными размерами мини-батчей, чтобы данные помещались в память GPU, при этом обучение было достаточно быстрым. Если вы запускаете этот код на своем собственном GPU, вы можете поэкспериментировать с настройкой размера мини-батча для ускорения обучения.

> **Примечание**: Известно, что некоторые версии драйверов NVidia не освобождают память после обучения модели. В этой тетради мы запускаем несколько примеров, и это может привести к исчерпанию памяти в определенных конфигурациях, особенно если вы проводите собственные эксперименты в рамках той же тетради. Если при старте обучения модели возникают странные ошибки, попробуйте перезапустить ядро тетради.


In [3]:
batch_size = 16
embed_size = 64

## Простой классификатор RNN

В случае простой RNN каждая рекуррентная единица представляет собой простую линейную сеть, которая принимает входной вектор и вектор состояния и выдает новый вектор состояния. В Keras это можно представить слоем `SimpleRNN`.

Хотя мы можем напрямую передавать one-hot кодированные токены в слой RNN, это не лучшая идея из-за их высокой размерности. Поэтому мы будем использовать слой embedding для уменьшения размерности векторов слов, за которым последует слой RNN, и, наконец, `Dense` классификатор.

> **Примечание**: В случаях, когда размерность не так высока, например, при использовании токенизации на уровне символов, может иметь смысл напрямую подавать one-hot кодированные токены в ячейку RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Примечание:** Здесь мы используем ненастроенный слой эмбеддингов для простоты, но для лучших результатов можно использовать предобученный слой эмбеддингов с помощью Word2Vec, как описано в предыдущем разделе. Было бы хорошим упражнением адаптировать этот код для работы с предобученными эмбеддингами.

Теперь давайте обучим нашу RNN. RNN в целом довольно сложно обучать, потому что после развёртывания ячеек RNN по длине последовательности количество слоёв, задействованных в обратном распространении ошибки, становится довольно большим. Поэтому нужно выбрать более низкую скорость обучения и обучать сеть на большем наборе данных, чтобы получить хорошие результаты. Это может занять довольно много времени, поэтому предпочтительно использовать GPU.

Чтобы ускорить процесс, мы будем обучать модель RNN только на заголовках новостей, опуская описание. Вы можете попробовать обучать с описанием и посмотреть, получится ли у вас обучить модель.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Обратите внимание**, что точность здесь, вероятно, будет ниже, так как мы тренируемся только на заголовках новостей.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Теперь, когда мы используем маскирование, мы можем обучить модель на всем наборе данных заголовков и описаний.

> **Примечание**: Вы заметили, что мы использовали векторизатор, обученный на заголовках новостей, а не на полном тексте статьи? Это потенциально может привести к тому, что некоторые токены будут игнорироваться, поэтому лучше переобучить векторизатор. Однако это может оказать очень незначительное влияние, поэтому для простоты мы останемся при предыдущем предварительно обученном векторизаторе.


## LSTM: Долгая краткосрочная память

Одна из основных проблем RNN — это **исчезающие градиенты**. RNN могут быть довольно длинными, и им может быть трудно распространять градиенты обратно до первого слоя сети во время обратного распространения ошибки. Когда это происходит, сеть не может изучать связи между отдалёнными токенами. Один из способов избежать этой проблемы — ввести **явное управление состоянием** с помощью **врат**. Две самые распространённые архитектуры, которые вводят врата — это **долгая краткосрочная память** (LSTM) и **элементы с управляемым переключателем** (GRU). Здесь мы рассмотрим LSTM.

![Изображение, показывающее пример ячейки долгой краткосрочной памяти](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Сеть LSTM организована подобным образом, как RNN, но между слоями передаются два состояния: собственное состояние $c$ и скрытый вектор $h$. В каждом блоке скрытый вектор $h_{t-1}$ комбинируется с входом $x_t$, и вместе они управляют тем, что происходит с состоянием $c_t$ и выходом $h_{t}$ через **врата**. Каждые врата имеют сигмоидальную активацию (выход в диапазоне $[0,1]$), что можно рассматривать как битовую маску, умножаемую на вектор состояния. В LSTM есть следующие врата (слева направо на картинке выше):
* **врата забывания**, которые определяют, какие компоненты вектора $c_{t-1}$ нужно забыть, а какие пропустить дальше.
* **входные врата**, которые определяют, сколько информации из входного вектора и предыдущего скрытого вектора следует включить в вектор состояния.
* **выходные врата**, которые берут новый вектор состояния и решают, какие его компоненты будут использованы для создания нового скрытого вектора $h_t$.

Компоненты состояния $c$ можно представить как флаги, которые можно включать и выключать. Например, когда мы встречаем имя *Алиса* в последовательности, мы предполагаем, что оно относится к женщине, и поднимаем флаг в состоянии, который говорит, что в предложении есть существительное женского рода. Когда далее встречаются слова *и Том*, мы поднимем флаг, указывающий на множественное число существительного. Таким образом, управляя состоянием, мы можем отслеживать грамматические свойства предложения.

> **Примечание**: Вот отличный ресурс для понимания внутреннего устройства LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) Кристофера Олаха.

Хотя внутренняя структура ячейки LSTM может выглядеть сложно, Keras скрывает эту реализацию внутри слоя `LSTM`, так что в приведённом выше примере нам нужно только заменить рекуррентный слой:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Обратите внимание**, что обучение LSTM также довольно медленное, и в начале обучения вы можете не заметить существенного увеличения точности. Возможно, вам придется продолжать обучение некоторое время, чтобы достичь хорошей точности.


## Двунаправленные и многослойные RNN

В наших примерах до сих пор рекуррентные сети работают от начала последовательности до конца. Это кажется нам естественным, потому что совпадает с направлением, в котором мы читаем или слушаем речь. Однако для сценариев, которые требуют случайного доступа к входной последовательности, более логично выполнять рекуррентные вычисления в обеих направлениях. RNN, которые позволяют вычисления в обоих направлениях, называются **двунаправленными** RNN, и их можно создать, обернув рекуррентный слой специальным слоем `Bidirectonal`.

> **Примечание**: слой `Bidirectional` создает две копии содержащегося слоя и устанавливает свойство `go_backwards` для одной из них в значение `True`, заставляя её работать в противоположном направлении вдоль последовательности.

Рекуррентные сети, как однонаправленные, так и двунаправленные, захватывают закономерности внутри последовательности и сохраняют их в векторных состояниях или возвращают как выходные данные. Как и в случае сверточных сетей, мы можем построить следующий рекуррентный слой за первым, чтобы захватывать закономерности более высокого уровня, построенные из более низкоуровневых, извлеченных первым слоем. Это подводит нас к понятию **многослойной RNN**, которая состоит из двух и более рекуррентных сетей, где выход предыдущего слоя передается на вход следующего слоя.

![Изображение, показывающее многослойную LSTM-RNN с долгосрочной памятью](../../../../../translated_images/ru/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Изображение из [этого замечательного поста](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) Фернандо Лопеса.*

Keras облегчает построение этих сетей, потому что вам просто нужно добавить больше рекуррентных слоев в модель. Для всех слоев, кроме последнего, нужно указать параметр `return_sequences=True`, чтобы слой возвращал все промежуточные состояния, а не только конечное состояние рекуррентного вычисления.

Давайте построим двуслойную двунаправленную LSTM для нашей задачи классификации.

> **Примечание**: этот код снова выполняется довольно долго, но показывает наивысшую точность, которую мы видели до сих пор. Так что, возможно, стоит подождать и увидеть результат.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## Рекуррентные нейронные сети для других задач

До сих пор мы сосредотачивались на использовании рекуррентных нейронных сетей для классификации последовательностей текста. Но они могут выполнять гораздо больше задач, таких как генерация текста и машинный перевод — мы рассмотрим эти задачи в следующем разделе.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
